In [1]:
import torch
import sqlite3
import pandas as pd
from pathlib import Path
from PIL import Image
from transformers import LlavaNextProcessor, LlavaNextForConditionalGeneration

/home/agrupa-lab/agrupa/IE_capstones/Omar/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16

print(f"Device: {DEVICE}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

BASE_DIR = Path("/home/agrupa-lab/agrupa")
DB_PATH = BASE_DIR / "agrupa.sqlite"
OUTPUT_DIR = Path("/home/agrupa-lab/agrupa/IE_capstones/Omar/outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

Device: cuda
GPU: NVIDIA GeForce RTX 5090
VRAM: 33.7 GB


In [3]:
conn = sqlite3.connect(DB_PATH)

query = """
    SELECT a.cat_no, a.titulo, a.autor, a.is_fauna, a.is_religious, a.century,
           i.file_path
    FROM artwork a
    INNER JOIN artwork_image i ON a.cat_no = i.cat_no
    WHERE substr(a.cat_no, 1, 1) = 'P'
"""
df = pd.read_sql(query, conn)
conn.close()

df['image_exists'] = df['file_path'].apply(lambda x: Path(x).exists())
df_ready = df[df['image_exists']].reset_index(drop=True)

print(f"Ready for LLaVA processing: {len(df_ready)}")
df_ready.head()

Ready for LLaVA processing: 6266


,cat_no,titulo,autor,is_fauna,is_religious,century,file_path,image_exists
0,P000002,El juicio de Paris,"Albani, Francesco",1,0,17th c.,/home/agrupa-lab/agrupa/scripts/urls/obras/P00...,True
1,P000006,Sagrada Familia y el cardenal Fernando de Medici,"Allori, Alessandro",1,1,16th c.,/home/agrupa-lab/agrupa/scripts/urls/obras/P00...,True
2,P000013,Rendición de Sevilla al rey san Fernando,"Flipart, Charles-Joseph",1,0,18th c.,/home/agrupa-lab/agrupa/scripts/urls/obras/P00...,True
3,P000014,"María Isabel de Borbón y Sajonia, infanta de N...","Ruta, Clemente",1,0,18th c.,/home/agrupa-lab/agrupa/scripts/urls/obras/P00...,True
4,P000015,La Anunciación,"Angelico, Fra",1,1,15th c.,/home/agrupa-lab/agrupa/scripts/urls/obras/P00...,True


In [4]:
print("Loading LLaVA model... (first run downloads ~15GB, be patient)")

MODEL_NAME = "llava-hf/llava-v1.6-mistral-7b-hf"

processor = LlavaNextProcessor.from_pretrained(MODEL_NAME)
model = LlavaNextForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
    device_map="auto"
)
model.eval()

print("LLaVA loaded successfully!")
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Loading LLaVA model... (first run downloads ~15GB, be patient)


The image processor of type `LlavaNextImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Loading weights: 100%|██████████| 687/687 [00:00<00:00, 813.50it/s] 


LLaVA loaded successfully!
VRAM used: 15.13 GB


In [5]:
PROMPT = "Describe this artwork in detail, including the people or animals depicted, their appearance, actions, social roles, and the overall mood of the scene."

def generate_description(image_path):
    try:
        image = Image.open(image_path).convert("RGB")
        
        formatted_prompt = f"[INST] <image>\n{PROMPT} [/INST]"
        
        inputs = processor(
            text=formatted_prompt,
            images=image,
            return_tensors="pt"
        ).to(DEVICE)
        
        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=300,
                do_sample=False
            )
        
        full_text = processor.decode(output[0], skip_special_tokens=True)
        description = full_text.split("[/INST]")[-1].strip()
        return description
    
    except Exception as e:
        return f"ERROR: {str(e)}"

In [6]:
sample = df_ready.sample(5, random_state=42)

print("Running LLaVA smoke test on 5 artworks...\n")
for _, row in sample.iterrows():
    desc = generate_description(row['file_path'])
    print(f"Artwork: {row['titulo']}")
    print(f"Author:  {row['autor']}")
    print(f"Description: {desc}")
    print("-" * 60)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Running LLaVA smoke test on 5 artworks...



Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Artwork: La reina María Luisa de Parma
Author:  Goya y Lucientes, Francisco de (Copia)
Description: The artwork is a portrait of a woman, rendered in a realistic style. The woman is depicted in a three-quarter view, facing to the left of the canvas. She is wearing a dress with a pattern of blue and black, adorned with a large feather in her hat, which is blue with a white plume. The dress has a high collar and long sleeves, suggesting a formal or historical attire.

The woman is holding a fan in her right hand, which is raised slightly above her waist. Her expression is serene, with a slight smile on her face. The background of the painting is dark, with hints of green and gold, which could suggest a luxurious setting or a room with richly colored drapery.

The overall mood of the scene is one of elegance and refinement. The woman's attire and the setting suggest a sense of sophistication and social status. The realistic style of the painting, combined with the attention to detail in t

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Artwork: Sagrada Familia
Author:  Cambiaso, Luca
Description: The artwork is a captivating oil painting that captures a tender moment between a woman and two children. The woman, dressed in a vibrant red dress, is seated on a blue cushion. She is holding a child in her arms, while another child is playfully kissing her cheek. The background is a dark red, providing a stark contrast to the subjects in the foreground. The overall mood of the scene is one of warmth and affection, as the characters engage in a moment of familial love. The painting is a beautiful representation of the Baroque style, characterized by its dramatic use of light and shadow, and its emphasis on emotion and movement.
------------------------------------------------------------


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Artwork: San Ambrosio imponiendo el hábito a san Agustín
Author:  García Hidalgo, José
Description: The artwork is a captivating oil painting that transports us to a medieval setting. The scene unfolds in a grand hall, characterized by its high arched ceiling and large windows that allow natural light to illuminate the room. The walls, painted in a dark hue, serve as a stark contrast to the white stone floor.

The painting is teeming with life, featuring a group of people engaged in various activities. Some are engaged in conversation, their faces turned towards each other in a display of camaraderie. Others are engrossed in their own world, their attention drawn to the intricate details of the room.

The figures are adorned in medieval clothing, adding to the authenticity of the scene. The colors used in the painting are predominantly dark, with the exception of the red and gold robes worn by some of the figures, which stand out against the otherwise somber palette.

The overall mood 

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Artwork: Vieja mesándose los cabellos
Author:  Massys, Jan (Círculo de)
Description: The artwork portrays an elderly woman, her face etched with the passage of time. Her hair, a mix of gray and white, frames her face, adding to her aged appearance. She is dressed in a white blouse, which contrasts with her green skirt, and her hands are gently placed on her head, suggesting a moment of contemplation or perhaps a gesture of frustration.

The background is a dark brown color, providing a stark contrast to the woman's white blouse and green skirt. This contrast further emphasizes the woman as the focal point of the artwork.

The overall mood of the scene is one of introspection and perhaps a hint of melancholy. The woman's actions and the overall composition of the artwork suggest a narrative that is open to interpretation. The artist has skillfully used color and light to create a sense of depth and dimension, adding to the overall impact of the piece.
-----------------------------------

After conducting a test with 5 artwork images, I observe that the generated descriptions are quite detailed and describe the artworks in a way that captures the elements of the image. Therefore, I will now be designing the pipeline to extract the descriptions for all artworks that have images. 

In [7]:
import time
import json
from tqdm import tqdm

def run_full_pipeline(df, batch_size=1, save_every=50):
    results = []
    errors = []
    
    output_file = OUTPUT_DIR / "llava_descriptions.csv"
    
    # Resume from checkpoint if exists
    if output_file.exists():
        existing = pd.read_csv(output_file)
        processed_ids = set(existing['cat_no'].tolist())
        df = df[~df['cat_no'].isin(processed_ids)].reset_index(drop=True)
        results = existing.to_dict('records')
        print(f"Resuming from checkpoint: {len(processed_ids)} already processed")
    else:
        processed_ids = set()
    
    print(f"Artworks to process: {len(df)}")
    
    start_time = time.time()
    
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        desc = generate_description(row['file_path'])
        
        result = {
            'cat_no': row['cat_no'],
            'titulo': row['titulo'],
            'autor': row['autor'],
            'is_fauna': row['is_fauna'],
            'is_religious': row['is_religious'],
            'century': row['century'],
            'llava_description': desc,
            'timestamp': pd.Timestamp.now().isoformat()
        }
        
        if desc.startswith("ERROR"):
            errors.append(result)
        
        results.append(result)
        
        # Save checkpoint every N artworks
        if len(results) % save_every == 0:
            pd.DataFrame(results).to_csv(output_file, index=False)
            print(f"Checkpoint saved: {len(results)} processed")
    
    # Final save
    results_df = pd.DataFrame(results)
    results_df.to_csv(output_file, index=False)
    
    elapsed = time.time() - start_time
    print(f"\nDone! Processed {len(results)} artworks in {elapsed/3600:.1f} hours")
    print(f"Errors: {len(errors)}")
    print(f"Saved to: {output_file}")
    
    return results_df

In [ ]:
# Time a single image to estimate total time
import time

sample_row = df_ready.iloc[0]
start = time.time()
_ = generate_description(sample_row['file_path'])
elapsed = time.time() - start

total_estimated = elapsed * len(df_ready)
print(f"Time per image: {elapsed:.1f}s")
print(f"Total artworks: {len(df_ready)}")
print(f"Estimated total time: {total_estimated/3600:.1f} hours")

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Time per image: 4.5s
Total artworks: 6266
Estimated total time: 7.8 hours


: 